# <font color="#418FDE" size="6.5" uppercase>**Tree Modeling**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Fit decision tree classifiers and regressors for nonlinear prediction tasks. 
- Configure split criteria, depth, leaf-size, and weighting controls. 
- Visualize and interpret fitted tree rules and predictions. 


## **1. Tree Estimators**

### **1.1. Classification Trees**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_01_01.jpg?v=1784006993" width="250">



>* Predicts categories using feature-based decision rules
>* Captures nonlinear thresholds, interactions, and context

>* Splits create purer class groups
>* Trees adapt rules to different data regions

>* Leaf class proportions guide predictions
>* Balance interpretability with generalization



In [ ]:
#@title Python Code - Classification Trees

# Fit a classification tree on iris data.
# Show nonlinear class boundaries from learned splits.
# Report accuracy and visualize the decision regions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load two flower measurements for easy plotting.
iris = load_iris()
feature_names = iris.feature_names[2:4]

X = iris.data[:, 2:4]
y = iris.target

# Validate the small tabular classification dataset.
if X.shape != (150, 2) or y.shape[0] != 150:
    raise ValueError("Expected 150 iris rows and two selected features.")

# Split data while preserving class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Limit depth so the tree stays interpretable.
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

# Evaluate predictions on unseen test data.
y_pred = tree.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Test accuracy: {accuracy:.2f}")
print(f"Tree depth: {tree.get_depth()}, leaves: {tree.get_n_leaves()}")

# Build a grid to show predicted regions.
x_min = X[:, 0].min() - 0.4
x_max = X[:, 0].max() + 0.4

y_min = X[:, 1].min() - 0.4
y_max = X[:, 1].max() + 0.4

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250)
)

# Predict the class for every grid point.
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_predictions = tree.predict(grid_points).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, grid_predictions, alpha=0.25, cmap="viridis")

scatter = ax.scatter(
    X_test[:, 0], X_test[:, 1], c=y_test, cmap="viridis", edgecolor="black"
)

ax.set_title("Decision Tree Classification Regions")
ax.set_xlabel(feature_names[0])
ax.set_ylabel(feature_names[1])

ax.legend(
    scatter.legend_elements()[0], iris.target_names, title="True class"
)
plt.show()



### **1.2. Regression Trees**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_01_02.jpg?v=1784006995" width="250">



>* Predict continuous values using decision tree splits
>* Leaf predictions average similar training examples

>* Captures nonlinear effects and feature interactions
>* Builds local rules for complex patterns

>* Trees make stepwise predictions within leaves
>* Balance depth, accuracy, stability, and interpretability



In [ ]:
#@title Python Code - Regression Trees

# Fit a regression tree for numeric prediction.
# Show stepwise predictions from learned tree leaves.
# Compare training and test accuracy simply.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Create a small nonlinear regression dataset.
rng = np.random.default_rng(42)
area = np.linspace(40, 220, 120)
noise = rng.normal(0, 18, size=area.shape)
price = 80 + 1.4 * area + 45 * (area > 130) + noise

# Use one feature, measured in square meters.
X = area.reshape(-1, 1)
y = price

# Check that features and targets align.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target values.")

# Split data before fitting the tree.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Limit depth so the tree stays interpretable.
tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=6, random_state=42)
tree.fit(X_train, y_train)

# Evaluate predictions on seen and unseen data.
train_mae = mean_absolute_error(y_train, tree.predict(X_train))
test_mae = mean_absolute_error(y_test, tree.predict(X_test))

print("scikit-learn version:", sklearn.__version__)
print("Tree depth:", tree.get_depth(), "leaves:", tree.get_n_leaves())
print("Train MAE:", round(train_mae, 1), "thousand dollars")
print("Test MAE:", round(test_mae, 1), "thousand dollars")

# Predict across a smooth grid to reveal steps.
area_grid = np.linspace(40, 220, 300).reshape(-1, 1)
predicted_grid = tree.predict(area_grid)

# Plot data points and the tree prediction steps.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train[:, 0], y_train, s=25, alpha=0.7, label="training homes")
ax.plot(area_grid[:, 0], predicted_grid, color="red", linewidth=2, label="tree prediction")
ax.set_title("Regression tree predictions are stepwise")
ax.set_xlabel("Home area (square meters)")
ax.set_ylabel("Price (thousand dollars)")
ax.legend()
plt.show()



### **1.3. Multioutput Tree Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_01_03.jpg?v=1784006997" width="250">



>* Predict several related targets at once
>* Shared splits improve all outputs together

>* One tree predicts multiple class labels
>* Shared splits balance related target goals

>* Predict multiple numeric outcomes per leaf
>* Check each output’s performance separately



In [ ]:
#@title Python Code - Multioutput Tree Models

# This example fits one multioutput regression tree.
# One tree predicts two related numeric targets.
# The plot compares true and predicted target pairs.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

# Create one feature with two related nonlinear targets.
rng = np.random.default_rng(42)
x = np.linspace(0, 10, 160).reshape(-1, 1)

# Each row has two target values to predict together.
noise = rng.normal(0, 0.15, size=(160, 2))
y = np.column_stack((np.sin(x[:, 0]), np.cos(x[:, 0]))) + noise

# Check that every sample has exactly two targets.
if y.shape[1] != 2:
    raise ValueError("Expected two target columns for multioutput regression.")

# Split features and both targets using the same rows.
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Fit one decision tree that predicts both targets.
tree = DecisionTreeRegressor(max_depth=4, min_samples_leaf=5, random_state=42)
tree.fit(X_train, y_train)

# Predict a two-number vector for each test sample.
y_pred = tree.predict(X_test)
r2_each_target = r2_score(y_test, y_pred, multioutput="raw_values")

print("scikit-learn version:", sklearn.__version__)
print("Prediction shape means rows by targets:", y_pred.shape)
print("R2 for sine target:", round(r2_each_target[0], 3))
print("R2 for cosine target:", round(r2_each_target[1], 3))

# Plot target one against target two for true and predicted pairs.
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_test[:, 0], y_test[:, 1], label="true pairs", alpha=0.75)
ax.scatter(y_pred[:, 0], y_pred[:, 1], label="tree predictions", alpha=0.75)

ax.set_title("Multioutput Decision Tree Predictions")
ax.set_xlabel("Target 1: sine value")
ax.set_ylabel("Target 2: cosine value")
ax.legend()
plt.show()



## **2. Split Controls**

### **2.1. Classification Split Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_02_01.jpg?v=1784007004" width="250">



>* Split criteria choose the best tree questions
>* Better splits create clearer class groups

>* Gini and entropy measure class impurity differently
>* Criteria can favor broad or distinctive splits

>* Match criteria to goals and error costs
>* Evaluate splits within the full modeling strategy



In [ ]:
#@title Python Code - Classification Split Criteria

# Compare split criteria in a classifier.
# Gini and entropy measure class impurity differently.
# The printed results show practical differences.

import sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load a small packaged classification dataset.
wine = load_wine()
X = wine.data
y = wine.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split once so both criteria see identical data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Fit two trees that differ only in split criterion.
criteria = ["gini", "entropy"]
results = []

for criterion in criteria:
    tree = DecisionTreeClassifier(
        criterion=criterion, max_depth=3, random_state=42
    )
    tree.fit(X_train, y_train)
    predictions = tree.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    first_feature = wine.feature_names[tree.tree_.feature[0]]
    results.append((criterion, accuracy, first_feature, tree.get_depth()))

print("scikit-learn version:", sklearn.__version__)
print("Criterion | Accuracy | Root split feature | Depth")
for criterion, accuracy, first_feature, depth in results:
    print(criterion, "|", round(accuracy, 3), "|", first_feature, "|", depth)



### **2.2. Regression Split Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_02_02.jpg?v=1784007006" width="250">



>* Split numeric targets into consistent groups
>* Choose splits that reduce prediction error

>* Squared error prioritizes reducing large prediction mistakes
>* Outliers can strongly influence resulting splits

>* Absolute error handles outliers more robustly
>* Balance criteria with depth and leaf controls



In [ ]:
#@title Python Code - Regression Split Criteria

# Compare regression tree split criteria.
# Squared error reacts more to outliers.
# Absolute error can give steadier predictions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.tree import DecisionTreeRegressor

# Build a small nonlinear regression dataset with outliers.
rng = np.random.default_rng(42)
x_values = np.linspace(0, 10, 80)

# The target has a smooth pattern plus small noise.
y_values = 2.0 * np.sin(x_values) + 0.4 * x_values
y_values = y_values + rng.normal(0, 0.35, size=x_values.shape)

# Add a few unusually high target values.
outlier_positions = np.array([12, 38, 65])
y_values[outlier_positions] = y_values[outlier_positions] + 7.0

# Scikit-learn expects a two-dimensional feature matrix.
features = x_values.reshape(-1, 1)
if features.shape[0] != y_values.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Fit two trees that differ only in regression split criterion.
squared_tree = DecisionTreeRegressor(
    criterion="squared_error", max_depth=3, random_state=42
)
squared_tree.fit(features, y_values)

absolute_tree = DecisionTreeRegressor(
    criterion="absolute_error", max_depth=3, random_state=42
)
absolute_tree.fit(features, y_values)

# Predict on a smooth grid to compare the learned step functions.
grid = np.linspace(0, 10, 300).reshape(-1, 1)
squared_predictions = squared_tree.predict(grid)
absolute_predictions = absolute_tree.predict(grid)

# Print concise facts that connect criteria to tree behavior.
print("scikit-learn version:", sklearn.__version__)
print("Squared-error leaves:", squared_tree.get_n_leaves())
print("Absolute-error leaves:", absolute_tree.get_n_leaves())

# Plot the data and both fitted regression trees.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_values, y_values, s=28, color="gray", label="training data")
ax.scatter(x_values[outlier_positions], y_values[outlier_positions], s=70,
           color="red", label="outliers")
ax.plot(grid.ravel(), squared_predictions, color="blue",
        label="criterion='squared_error'")
ax.plot(grid.ravel(), absolute_predictions, color="green",
        label="criterion='absolute_error'")

# Label the single figure for beginner interpretation.
ax.set_title("Regression tree split criteria with outliers")
ax.set_xlabel("Feature value")
ax.set_ylabel("Numeric target")
ax.legend()
plt.show()



### **2.3. Depth and Leaf Controls**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_02_03.jpg?v=1784007008" width="250">



>* Limit tree depth to control complexity
>* Balance underfitting against overfitting risk

>* Minimum leaf sizes prevent fragile predictions
>* Larger leaves create stable, interpretable groups

>* Tune depth and leaves together
>* Match settings to data and interpretability



In [ ]:
#@title Python Code - Depth and Leaf Controls

# Compare tree depth and leaf-size controls.
# Smaller controls make simpler decision rules.
# Test accuracy shows the generalization tradeoff.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Create a small nonlinear classification dataset.
features, target = make_moons(n_samples=500, noise=0.28, random_state=42)

# Check that the generated data has the expected shape.
if features.shape != (500, 2):
    raise ValueError("Expected 500 rows and 2 features.")

# Split data so accuracy is measured on unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

# Try three depth and leaf-size settings.
settings = [
    (2, 1, "shallow"),
    (5, 10, "balanced"),
    (None, 1, "very flexible"),
]

print("scikit-learn version:", sklearn.__version__)
print("setting | depth | min leaf | leaves | train acc | test acc")

# Fit one tree for each control setting.
results = []
for max_depth, min_leaf, label in settings:
    tree = DecisionTreeClassifier(
        max_depth=max_depth, min_samples_leaf=min_leaf, random_state=42
    )
    tree.fit(X_train, y_train)
    train_accuracy = accuracy_score(y_train, tree.predict(X_train))
    test_accuracy = accuracy_score(y_test, tree.predict(X_test))
    results.append((label, tree.get_n_leaves(), train_accuracy, test_accuracy))
    depth_text = "None" if max_depth is None else str(max_depth)
    print(label, "|", depth_text, "|", min_leaf, "|", tree.get_n_leaves(), "|", round(train_accuracy, 3), "|", round(test_accuracy, 3))

# Plot test accuracy to compare generalization.
labels = [row[0] for row in results]
test_scores = [row[3] for row in results]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, test_scores, color=["#8ecae6", "#219ebc", "#ffb703"])
ax.set_title("Depth and Leaf Controls Affect Test Accuracy")
ax.set_xlabel("Tree control setting")
ax.set_ylabel("Test accuracy")
ax.set_ylim(0, 1)
plt.show()



## **3. Tree Interpretation**

### **3.1. Class Weight Effects**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_03_01.jpg?v=1784006999" width="250">



>* Class weights change split and leaf decisions
>* Tree rules reflect weighted modeling priorities

>* Weighted classes can override raw leaf counts
>* Interpret rules using data and priorities

>* Weights can add splits for priority cases
>* Interpret rules through error trade-offs



In [ ]:
#@title Python Code - Class Weight Effects

# Compare weighted and unweighted decision tree predictions.
# Class weights can change leaf decisions.
# The plot shows shifted decision regions.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

# Create an imbalanced two-feature classification dataset.
features, labels = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    weights=[0.9, 0.1],
    class_sep=0.8,
    random_state=42,
)

# Validate the small dataset before modeling.
if features.shape != (300, 2) or labels.shape != (300,):
    raise ValueError("Expected 300 rows, two features, and one label per row.")

# Fit two trees that differ only in class weighting.
unweighted_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
weighted_tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight={0: 1, 1: 8},
    random_state=42,
)

unweighted_tree.fit(features, labels)
weighted_tree.fit(features, labels)

# Compare how often each tree predicts the minority class.
unweighted_minority_rate = unweighted_tree.predict(features).mean()
weighted_minority_rate = weighted_tree.predict(features).mean()

print(f"scikit-learn version: {sklearn_version}")
print(f"Actual minority class rate: {labels.mean():.2f}")
print(f"Unweighted tree predicts minority rate: {unweighted_minority_rate:.2f}")
print(f"Weighted tree predicts minority rate: {weighted_minority_rate:.2f}")

# Build a grid to visualize the weighted tree decision regions.
x_min = features[:, 0].min() - 0.5
x_max = features[:, 0].max() + 0.5
y_min = features[:, 1].min() - 0.5
y_max = features[:, 1].max() + 0.5

x_values = np.linspace(x_min, x_max, 180)
y_values = np.linspace(y_min, y_max, 180)
grid_x, grid_y = np.meshgrid(x_values, y_values)
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

weighted_regions = weighted_tree.predict(grid_points).reshape(grid_x.shape)

# Plot the weighted tree regions and the training points.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(grid_x, grid_y, weighted_regions, alpha=0.25, cmap="coolwarm")
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=labels,
    cmap="coolwarm",
    edgecolor="black",
    s=35,
)

ax.set_title("Weighted Decision Tree: Minority Class Gets More Influence")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(*scatter.legend_elements(), title="Class", loc="upper right")
plt.show()



### **3.2. Handling Missing Values**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_03_02.jpg?v=1784007001" width="250">



>* Missing split values affect prediction paths
>* Handle missingness before or during modeling

>* Missingness can carry real-world meaning
>* Root-level missingness can shape many predictions

>* Explain how missing values were handled
>* Test prediction stability across handling choices



In [ ]:
#@title Python Code - Handling Missing Values

# This example shows missing values in tree interpretation.
# Imputation choices can change a tree path.
# The printed paths reveal different explanations.

import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

# A tiny loan dataset keeps the tree easy to inspect.
loan_data = pd.DataFrame(
    {
        "income_k": [35, 42, 48, 55, 62, 70, 85, 95],
        "debt_k": [28, 25, 22, 20, 18, 15, 12, 10],
    }
)

# One means high risk, and zero means lower risk.
risk_label = np.array([1, 1, 1, 0, 0, 0, 0, 0])
new_applicant = pd.DataFrame({"income_k": [np.nan], "debt_k": [24]})

# Validate the small teaching dataset before modeling.
if loan_data.shape != (8, 2):
    raise ValueError("Expected eight rows and two features.")

# Fit one shallow tree after median imputation.
median_imputer = SimpleImputer(strategy="median")
median_features = median_imputer.fit_transform(loan_data)
median_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
median_tree.fit(median_features, risk_label)

# Fit the same tree after constant imputation.
constant_imputer = SimpleImputer(strategy="constant", fill_value=0)
constant_features = constant_imputer.fit_transform(loan_data)
constant_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
constant_tree.fit(constant_features, risk_label)

# Convert each tree path into beginner-friendly rule text.
def describe_path(tree, feature_names, row):
    node_ids = tree.decision_path(row).indices
    rules = []

    for node_id in node_ids:
        feature_id = tree.tree_.feature[node_id]
        if feature_id < 0:
            continue

        feature_name = feature_names[feature_id]
        threshold = tree.tree_.threshold[node_id]
        value = row[0, feature_id]

        direction = "left" if value <= threshold else "right"
        rule = f"{feature_name}={value:.1f} goes {direction} at {threshold:.1f}"
        rules.append(rule)

    return " | ".join(rules)

# Compare how the originally missing income is explained.
median_row = median_imputer.transform(new_applicant)
constant_row = constant_imputer.transform(new_applicant)
median_prediction = median_tree.predict(median_row)[0]
constant_prediction = constant_tree.predict(constant_row)[0]

print(f"scikit-learn version: {sklearn.__version__}")
print("Original new applicant: income_k is missing, debt_k is 24.")
print(f"Median imputation prediction: risk={median_prediction}")
print(describe_path(median_tree, loan_data.columns, median_row))
print(f"Constant imputation prediction: risk={constant_prediction}")
print(describe_path(constant_tree, loan_data.columns, constant_row))



### **3.3. Visualizing Tree Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_A/image_03_03.jpg?v=1784007002" width="250">



>* Tree diagrams show feature-based decision paths
>* Leaves explain final class or value predictions

>* Root splits create increasingly specific groups
>* Sample counts show pattern reliability

>* Assess tree clarity and practical usefulness
>* Check splits for meaning or leakage



In [ ]:
#@title Python Code - Visualizing Tree Structure

# Visualize a small fitted decision tree.
# Read each node as a question.
# Trace one prediction from root to leaf.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

# Load a small built-in classification dataset.
iris = load_iris()
X = iris.data[:, [2, 3]]
y = iris.target

# Validate the two-feature teaching dataset.
if X.shape[0] != y.shape[0] or X.shape[1] != 2:
    raise ValueError("Expected matching rows and exactly two features.")

# Fit a shallow tree so the rules stay readable.
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X, y)

# Trace one flower through the fitted tree.
sample_index = 100
sample = X[sample_index:sample_index + 1]
predicted_class = tree.predict(sample)[0]

# Print concise context for interpreting the diagram.
print("scikit-learn version:", sklearn.__version__)
print("Example flower petal length:", round(sample[0, 0], 2), "cm")
print("Example flower petal width:", round(sample[0, 1], 2), "cm")
print("Predicted species:", iris.target_names[predicted_class])

# Draw the tree as readable decision rules.
fig, ax = plt.subplots(figsize=(11, 6))
plot_tree(
    tree,
    feature_names=["petal length (cm)", "petal width (cm)"],
    class_names=list(iris.target_names),
    filled=True,
    rounded=True,
    ax=ax,
)

# The root is the first question, and leaves are predictions.
ax.set_title("Decision Tree Structure for Iris Petal Measurements")
ax.set_xlabel("Follow left for true conditions and right for false conditions")
ax.set_ylabel("Tree depth: more specific rules appear lower")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Tree Modeling**</font>


In this lecture, you learned to:
- Fit decision tree classifiers and regressors for nonlinear prediction tasks. 
- Configure split criteria, depth, leaf-size, and weighting controls. 
- Visualize and interpret fitted tree rules and predictions. 

In the next Lecture (Lecture B), we will go over 'Tree Regularization'